In [1]:
import pandas as pd
import re

#email_data = pd.read_csv('C:/Users/User/OneDrive - Asia Pacific University/APU Final Year Project/FYP Email Project Documents/Email Dataset Python/Email Qwen Priority Dataset/priority_synthetic_balanced_full_dataset.csv')
email_data = pd.read_csv('C:/Users/User/OneDrive - Asia Pacific University/APU Final Year Project/FYP Email Project Documents/Email Dataset Python/Email Qwen Priority Dataset/realistic_priority_email_dataset_full_format.csv')

In [2]:
email_data.info(memory_usage='deep') 
print("Number of Rows and Columns: ", email_data.shape) 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 108296 entries, 0 to 108295
Data columns (total 13 columns):
 #   Column           Non-Null Count   Dtype 
---  ------           --------------   ----- 
 0   From             108296 non-null  object
 1   To               108296 non-null  object
 2   Subject          108296 non-null  object
 3   Message          108296 non-null  object
 4   Day              108296 non-null  object
 5   Date & Time      108296 non-null  object
 6   url_count        108296 non-null  int64 
 7   Cleaned_Subject  108296 non-null  object
 8   Cleaned_Message  108296 non-null  object
 9   Combined_Text    108296 non-null  object
 10  Subject_LLM      108296 non-null  object
 11  Message_LLM      108296 non-null  object
 12  Email_Priority   108296 non-null  object
dtypes: int64(1), object(12)
memory usage: 176.0 MB
Number of Rows and Columns:  (108296, 13)


In [ ]:
email_data

In [4]:
import re
import spacy

# Load spaCy model (make sure to run: python -m spacy download en_core_web_sm)
nlp = spacy.load("en_core_web_sm", disable=["ner", "parser"])

def preprocess_email_for_modeling(text, max_length=300):
    if not isinstance(text, str):
        return ""

    # --- Step 1: Clean Common Artifacts ---
    text = re.sub(r"(?is)-----.*?Subject:", "", text)  # Email chains
    text = re.sub(r'=3D', '=', text)  # Quoted-printable artifacts
    text = re.sub(r'=20', ' ', text)
    text = re.sub(r'=09', ' ', text)
    text = re.sub(r'=\s?', '', text)

    # Remove email signatures and greetings
    text = re.sub(r"(?im)^(hi team,|team,|hello,|hi,|dear\s+\w+,?)", "", text)
    text = re.sub(r"(?i)(thanks|regards|sincerely|best),?", "", text)

    # Remove URLs, emails, phone numbers
    text = re.sub(r"http[s]?://\S+", "", text)
    text = re.sub(r"\b\S+@\S+\b", "", text)
    text = re.sub(r"<< File:.*?>>", "", text)
    text = re.sub(r"\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}", "", text)

    # Normalize spacing and punctuation
    text = re.sub(r"[\r\n\t]+", " ", text)
    text = re.sub(r"\.\.+", ".", text)
    text = re.sub(r"[^a-zA-Z0-9.,!? ]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    text = text.lower()

    # --- Step 2: Tokenize + Lemmatize + Remove Stopwords ---
    doc = nlp(text)
    tokens = [
        token.lemma_.lower()
        for token in doc
        if not token.is_stop and not token.is_punct and not token.like_num and len(token) > 1
    ]

    # Truncate to 300 words max
    tokens = tokens[:max_length]

    return " ".join(tokens)

email_data['Cleaned_Message'] = email_data['Message'].apply(preprocess_email_for_modeling)

In [ ]:
email_data

# Subject Cleaning

In [5]:
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

nltk.download('punkt')
stop_words = set(stopwords.words('english'))

def clean_subject_for_modeling(text, remove_stopwords=True):
    if not isinstance(text, str):
        return ""
    
    # 1. Lowercase
    text = text.lower()
    
    # 2. Remove reply/forward indicators
    text = re.sub(r"(?i)^(fw:|fwd:|re:|\[.*?\])\s*", "", text)

    # 3. Remove standalone numbers (optional, or keep if part of context)
    text = re.sub(r"\b\d+\b", "", text)

    # 4. Remove symbols but keep useful ones like slashes or dashes
    text = re.sub(r"[^\w\s/-]", "", text)

    # 5. Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    # 6. Tokenize
    tokens = word_tokenize(text)

    # 7. Stopword filtering (optional for BERT)
    if remove_stopwords:
        tokens = [token for token in tokens if token not in stop_words and len(token) > 1]

    # 8. Rebuild subject line
    return " ".join(tokens)

email_data['Cleaned_Subject'] = email_data['Subject'].apply(lambda x: clean_subject_for_modeling(x, remove_stopwords=True))

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [6]:
# Count NaN, None, and empty string values in the 'Cleaned_Subject' column
nan_count = email_data['Cleaned_Subject'].isna().sum() + (email_data['Cleaned_Subject'] == '').sum()

# Print the result
print(f"Number of NaN or empty values in 'Cleaned_Subject': {nan_count}")

Number of NaN or empty values in 'Cleaned_Subject': 0


In [7]:
# Count NaN, None, and empty string values in the 'Cleaned_Subject' column
nan_count = email_data['Cleaned_Message'].isna().sum() + (email_data['Cleaned_Message'] == '').sum()

# Print the result
print(f"Number of NaN or empty values in 'Cleaned_Message': {nan_count}")

Number of NaN or empty values in 'Cleaned_Message': 0


In [8]:
# Combine Subject and Message into Combined_Text
email_data['Combined_Text'] = email_data['Cleaned_Subject'] + " " + email_data['Cleaned_Message']

# Final step: Remove accidental 'nan' (in case subject or message was missing) and extra spaces
email_data['Combined_Text'] = email_data['Combined_Text'].str.replace(r'\b[nN][aA][nN]\b', '', regex=True).str.strip()

In [9]:
# Count NaN, None, and empty string values in the 'Cleaned_Subject' column
nan_count = email_data['Combined_Text'].isna().sum() + (email_data['Combined_Text'] == '').sum()

# Print the result
print(f"Number of NaN or empty values in 'Combined_Text': {nan_count}")

Number of NaN or empty values in 'Combined_Text': 0


In [ ]:
email_data

# Subject LLM Cleaning

In [10]:
import pandas as pd
import re
from bs4 import BeautifulSoup

def clean_email_data(email_data):
    """
    Function to clean 'Subject' and 'Message' columns:
    - Strips leading/trailing spaces
    - Removes special characters and symbols
    - Removes HTML tags
    - Stores cleaned versions in new columns 'Subject_LLM' and 'Message_LLM'
    """
    
    def clean_text(text):
        """
        Helper function to clean the text by:
        - Stripping leading/trailing spaces
        - Removing special characters and symbols
        - Removing HTML tags
        - Handling non-string values
        """
        if not isinstance(text, str):  # Ensure the text is a string
            text = str(text) if text is not None else ""  # Convert non-string to an empty string if None

        # Step 1: Remove HTML tags
        text = BeautifulSoup(text, "html.parser").get_text()

        # Step 2: Remove special characters or symbols (allow letters, numbers, and spaces)
        text = re.sub(r'[^a-zA-Z0-9\s]', '', text)

        # Step 3: Strip leading and trailing spaces
        text = text.strip()

        return text

    # Step 1: Clean 'Subject' and 'Message' columns and store the cleaned text in new columns
    email_data['Subject_LLM'] = email_data['Subject'].apply(clean_text)
    email_data['Message_LLM'] = email_data['Message'].apply(clean_text)

    return email_data

# Clean the email data
email_data_cleaned = clean_email_data(email_data)

C:\Users\User\AppData\Local\Temp\ipykernel_5652\1068849092.py:26: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  text = BeautifulSoup(text, "html.parser").get_text()


In [ ]:
email_data_cleaned

In [12]:
# Save the DataFrame to a CSV file
email_data_cleaned.to_csv('cleaned_realistic_priority_email_dataset_full_format.csv', index=False)